# 01 — Data Preparation
**Project:** Disease Detection and Fungicide Treatment Recommendation for Rosa damascena

This notebook downloads and organises all raw datasets used in the project:
- Weather data for Krasnovo, Plovdiv Province (Open-Meteo API)
- Disease occurrence records for 4 Rosa pathogens (GBIF API)

Processed files are saved to `data/processed/` for use in subsequent notebooks.

In [1]:
import requests
import time
import cv2
import hashlib
import shutil
import yaml
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
BASE_DIR = Path("..")
RAW = BASE_DIR / "data" / "raw"
PROCESSED = BASE_DIR / "data" / "processed"

folders = [
    RAW / "weather",
    RAW / "gbif",
    RAW / "images",
    PROCESSED / "weather",
    PROCESSED / "images",
]
for f in folders:
    f.mkdir(parents=True, exist_ok=True)

print("Folder structure ready.")

Folder structure ready.


## Setup

All required directories exist under `data/raw/` and `data/processed/`. The notebook references the project root via `Path("..")` so all paths work regardless of the operating system

## Field Location — Krasnovo, Plovdiv Province

The following cell queries the Open-Meteo geocoding API to retrieve the exact coordinates for Krasnovo, Bulgaria. Results are kept for reference —
coordinates are hardcoded in the weather download cell below.

In [3]:
response = requests.get(
    "https://geocoding-api.open-meteo.com/v1/search",
    params={"name": "Krasnovo", "country_code": "BG", "count": 5, "language": "en"}
)

results = response.json().get("results", [])

for r in results:
    print(f"{r['name']}, {r.get('admin1', '')} | lat: {r['latitude']}, lon: {r['longitude']}, elevation: {r.get('elevation', 'N/A')} m")

Krasnovo, Plovdiv | lat: 42.46667, lon: 24.48333, elevation: 363.0 m
Krasnovo, Nizhny Novgorod Oblast | lat: 55.7399, lon: 42.7271, elevation: 159.0 m
Krasnovo, Vologda Oblast | lat: 59.83372, lon: 38.37893, elevation: 125.0 m
Krasnovo, Vologda Oblast | lat: 59.9158, lon: 38.6843, elevation: 127.0 m
Krasnovo, Vologda Oblast | lat: 58.97755, lon: 38.98736, elevation: 165.0 m


In [16]:
weather_path = RAW / "weather" / "krasnovo_weather_2015_2024.csv"

if weather_path.exists():
    weather_df = pd.read_csv(weather_path)
    weather_df["date"] = pd.to_datetime(weather_df["date"])
    print(f"Loaded from disk: {len(weather_df)} records")
else:
    LAT = 42.46667   
    LON = 24.48333

    response = requests.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params={
            "latitude": LAT,
            "longitude": LON,
            "start_date": "2015-01-01",
            "end_date": "2024-12-31",
            "daily": [
                "temperature_2m_max",
                "temperature_2m_min",
                "temperature_2m_mean",
                "precipitation_sum",
                "relative_humidity_2m_mean",
                "wind_speed_10m_max",
                "et0_fao_evapotranspiration"
            ],
            "timezone": "Europe/Sofia"
        }
    )

    data = response.json()
    weather_df = pd.DataFrame(data["daily"])
    weather_df.rename(columns={"time": "date"}, inplace=True)
    weather_df["date"] = pd.to_datetime(weather_df["date"])
    weather_df.to_csv(weather_path, index=False)
    print(f"Downloaded and saved: {len(weather_df)} records")

Loaded from disk: 3653 records


## Weather Data - Krasnovo, Plovdiv Province (2015-2024)

Downloaded 3,653 daily records from Open-Meteo (ERA5 reanalysis) with no missing values across all 7 variables. The dataset covers a full 10-year period including three leap years (2016, 2020, 2024).

Variables collected:
- **Temperature:** daily min, max and mean (°C) - key driver of fungal development
- **Precipitation:** daily total (mm) - wet conditions favour infection
- **Relative humidity:** daily mean (%) - critical for spore germination
- **Wind speed:** daily max (km/h) - affects spore dispersal
- **ET0:** reference evapotranspiration (mm) - indicator of atmospheric dryness

This data will be used in `07_weather_risk.ipynb` to model disease risk based on meteorological conditions.

In [ ]:
diseases = {
    "Diplocarpon rosae": 2583566,
    "Phragmidium mucronatum": 7789914,
    "Podosphaera pannosa": 5255308,
    "Peronospora sparsa": 3203846
}

for name, key in diseases.items():
    print(f"{name}: {key}")

In [21]:
gbif_path = RAW / "gbif" / "gbif_rosa_diseases_europe.csv"

if gbif_path.exists():
    gbif_df = pd.read_csv(gbif_path, low_memory=False)
    print(f"Loaded from disk: {len(gbif_df)} records")
else:
    all_gbif = []

    for species_name, taxon_key in diseases.items():
        print(f"Downloading: {species_name}...")
        params = {"taxonKey": taxon_key, "limit": 300, "offset": 0}
        species_records = []

        while True:
            r = requests.get("https://api.gbif.org/v1/occurrence/search", params=params)
            batch = r.json()
            results = batch.get("results", [])
            species_records.extend(results)

            if batch.get("endOfRecords", True):
                break

            params["offset"] += params["limit"]
            time.sleep(0.5)

        print(f"  → {len(species_records)} records")

        for rec in species_records:
            rec["query_species"] = species_name

        all_gbif.extend(species_records)

    gbif_df = pd.DataFrame(all_gbif)
    gbif_df.to_csv(gbif_path, index=False)
    print(f"\nDownloaded and saved: {len(gbif_df)} records")

Downloading: Diplocarpon rosae...
  → 4436 records
Downloading: Phragmidium mucronatum...
  → 8128 records
Downloading: Podosphaera pannosa...
  → 5873 records
Downloading: Peronospora sparsa...
  → 490 records

Downloaded and saved: 18927 records


## GBIF Disease Occurrence Records — Europe

Downloaded 18,927 occurrence records across 4 target diseases:

| Disease | Records |
|---|---|
| Phragmidium mucronatum (rust) | 8,128 |
| Podosphaera pannosa (powdery mildew) | 5,873 |
| Diplocarpon rosae (black spot) | 4,436 |
| Peronospora sparsa (downy mildew) | 490 |
| **Total** | **18,927** |

Data covers European observations without geographic restriction. Rose rust accounts for 43% of all records, consistent with it being the most visible and widely reported rose disease. Downy mildew is underrepresented (2.6%) reflecting lower reporting rates globally.

In [23]:
print(gbif_df.shape)
print(gbif_df.columns.to_list())

(18927, 228)
['key', 'datasetKey', 'publishingOrgKey', 'installationKey', 'hostingOrganizationKey', 'publishingCountry', 'protocol', 'lastCrawled', 'lastParsed', 'crawlId', 'extensions', 'basisOfRecord', 'occurrenceStatus', 'classifications', 'taxonKey', 'kingdomKey', 'phylumKey', 'classKey', 'orderKey', 'familyKey', 'genusKey', 'speciesKey', 'acceptedTaxonKey', 'scientificName', 'scientificNameAuthorship', 'acceptedScientificName', 'kingdom', 'phylum', 'order', 'family', 'genus', 'species', 'genericName', 'specificEpithet', 'taxonRank', 'taxonomicStatus', 'dateIdentified', 'decimalLatitude', 'decimalLongitude', 'coordinateUncertaintyInMeters', 'continent', 'stateProvince', 'gadm', 'year', 'month', 'day', 'eventDate', 'startDayOfYear', 'endDayOfYear', 'issues', 'modified', 'lastInterpreted', 'references', 'license', 'isSequenced', 'identifiers', 'media', 'facts', 'relations', 'isInCluster', 'datasetName', 'recordedBy', 'identifiedBy', 'dnaSequenceID', 'geodeticDatum', 'class', 'country

In [22]:
cols = [
    "gbifID",
    "query_species",
    "decimalLatitude",
    "decimalLongitude",
    "year",
    "month",
    "day",
    "eventDate",
    "country",
    "stateProvince",
    "basisOfRecord",
    "coordinateUncertaintyInMeters"
]

gbif_clean = gbif_df[cols].copy()

print(gbif_clean.shape)
print(f"\nMissing values:\n{gbif_clean.isnull().sum()}")
print(f"\nRecords with coordinates: {gbif_clean[['decimalLatitude', 'decimalLongitude']].notna().all(axis=1).sum()}")
print(f"\nCountries:\n{gbif_clean['country'].value_counts().head(10)}")

(18927, 12)

Missing values:
gbifID                              0
query_species                       0
decimalLatitude                  7172
decimalLongitude                 7172
year                             2803
month                            3704
day                              5337
eventDate                        2771
country                          1680
stateProvince                    6718
basisOfRecord                       0
coordinateUncertaintyInMeters    9808
dtype: int64

Records with coordinates: 11755

Countries:
country
United Kingdom of Great Britain and Northern Ireland    4511
United States of America                                4419
Germany                                                 1312
Sweden                                                  1228
Switzerland                                              428
Estonia                                                  420
Canada                                                   359
Finland               

## GBIF Data — Initial Inspection

Reduced from 228 to 12 relevant columns. Key findings:

- 11,755 records (62%) have valid coordinates — usable for spatial analysis
- 7,172 records missing coordinates — will be excluded from map-based analysis
- UK and USA dominate the dataset (citizen science bias via iNaturalist)
- Bulgaria is nearly absent, reflecting low citizen science activity,
  not low disease prevalence

Missing values in year (2,803) and month (3,704) will limit temporal analysis for a subset of records.

In [24]:
gbif_coords = gbif_clean.dropna(subset=["decimalLatitude", "decimalLongitude"]).copy()

gbif_coords.to_csv(PROCESSED / "gbif_rosa_diseases_clean.csv", index=False)

print(f"Records with coordinates saved: {len(gbif_coords)}")
print(f"Dropped: {len(gbif_clean) - len(gbif_coords)} records")

Records with coordinates saved: 11755
Dropped: 7172 records


## GBIF Data - Saved
Filtered to 11,755 records with valid coordinates and saved to `data/processed/gbif_rosa_diseases_clean.csv`.
7,172 records without coordinates were dropped. Spatial analysis requires decimal latitude and longitude to place observation on a map, calculate distances between locations and match records to climate zones. Without these values the records cannot be positioned geographically and are therefore excluded from any map-based or distance-based analysis.